# AMP Designer — Layer 1: Data + Baseline Activity Classifier
**Stage 1 (in-silico), computational half.** This notebook builds the first working piece of the pipeline: a model that predicts, from a peptide's amino-acid sequence alone, whether it's likely to be an antimicrobial peptide (AMP) — i.e. likely to kill bacteria.

This is a *standalone result*. The generator, toxicity filter, and full ranking loop come in later notebooks.

**Honest framing:** the output is a *prediction with uncertainty*, not proof. No software confirms a kill — that needs a wet lab (Stage 3).

**Data**
- **Positives (AMPs):** GRAMPA — a consolidated CSV merging DBAASP, DRAMP, YADAMP, APD, DADP. ~6,760 unique peptides, ~51k MIC measurements (MIC = minimum inhibitory concentration, the real lab potency number; lower = more potent). We keep the MIC values for later layers.
- **Negatives (non-AMPs):** AMPlify's UniProt-derived non-antimicrobial proteins. *Why we need these:* AMP databases contain almost only things that work, so a model trained on positives alone learns nothing. The field's standard fix is biological decoys from UniProt. **This is the main swappable design choice** — see the limitations cell.

Runs free on Kaggle/Colab (CPU is fine; trains in well under a minute).

In [ ]:
# If on a fresh Kaggle/Colab runtime, uncomment:
# !pip install scikit-learn pandas numpy matplotlib joblib -q
import pandas as pd, numpy as np, joblib
from collections import Counter
import matplotlib.pyplot as plt
np.random.seed(42)
CANON = "ACDEFGHIKLMNPQRSTVWY"; CANSET = set(CANON)
MINLEN, MAXLEN = 5, 50   # AMPs are short; brief caps at ~52

## 1. Get the data
Downloaded at runtime from public GitHub mirrors (needs internet ON in Kaggle/Colab).

In [ ]:
import urllib.request, os
FILES = {
 'grampa.csv': 'https://raw.githubusercontent.com/zswitten/Antimicrobial-Peptides/master/data/grampa.csv',
 'neg_train.fa': 'https://raw.githubusercontent.com/BirolLab/AMPlify/master/data/AMPlify_non_AMP_train_balanced.fa',
 'neg_test.fa':  'https://raw.githubusercontent.com/BirolLab/AMPlify/master/data/AMPlify_non_AMP_test_balanced.fa',
}
for fn,url in FILES.items():
    if not os.path.exists(fn):
        urllib.request.urlretrieve(url, fn); print('downloaded', fn)
    else: print('exists', fn)

## 2. Understand the data before modelling
Good scientific habit: look at it first.

In [ ]:
g = pd.read_csv('grampa.csv')
g['sequence'] = g['sequence'].astype(str).str.upper()
g['seqlen'] = g['sequence'].str.len()
print('rows (MIC measurements):', len(g))
print('unique peptides         :', g['sequence'].nunique())
print('sources                 :', g['database'].value_counts().to_dict())
print('MIC unit                :', g['unit'].unique(), '(value column = log10 of MIC in uM)')

fig, ax = plt.subplots(1, 2, figsize=(11,3.4))
ax[0].hist(g['value'], bins=60); ax[0].set_title('log10(MIC, uM) — lower = more potent'); ax[0].set_xlabel('log10 MIC')
ax[1].hist(g['seqlen'].clip(upper=80), bins=60); ax[1].set_title('peptide length'); ax[1].set_xlabel('residues')
plt.tight_layout(); plt.show()

## 3. Build the labelled dataset
- **Positives:** each unique GRAMPA peptide, keeping its *best* (minimum) MIC across all strains — broad-spectrum potency. Cleaned to the 20 standard amino acids, length 5–50.
- **Negatives:** UniProt non-AMPs, same cleaning.
- **Length-matched + balanced 1:1** so the model can't cheat by just learning sequence length, and we remove any sequence overlap to prevent leakage.

In [ ]:
def clean(seqs):
    out=[]
    for s in seqs:
        s=str(s).strip().upper()
        if s and all(ch in CANSET for ch in s) and MINLEN<=len(s)<=MAXLEN:
            out.append(s)
    return list(dict.fromkeys(out))

def read_fasta(p):
    seqs, cur = [], ''
    for line in open(p):
        line=line.strip()
        if line.startswith('>'):
            if cur: seqs.append(cur); cur=''
        else: cur+=line
    if cur: seqs.append(cur)
    return seqs

minmic = g.groupby('sequence')['value'].min()        # keep for later layers (MIC regression)
pos = clean(minmic.index.tolist())
neg = clean(read_fasta('neg_train.fa') + read_fasta('neg_test.fa'))
neg = [s for s in neg if s not in set(pos)]           # no leakage

# length-match in 5-residue bins, balance 1:1
def lb(s): return len(s)//5
from collections import defaultdict
pb, nb = defaultdict(list), defaultdict(list)
for s in pos: pb[lb(s)].append(s)
for s in neg: nb[lb(s)].append(s)
sp, sn = [], []
for b in set(list(pb)+list(nb)):
    k = min(len(pb[b]), len(nb[b]))
    if k:
        sp += list(np.random.choice(pb[b], k, replace=False))
        sn += list(np.random.choice(nb[b], k, replace=False))
seqs = sp + sn
y = np.array([1]*len(sp) + [0]*len(sn))
print(f'positives available={len(pos)}, negatives available={len(neg)}')
print(f'balanced + length-matched: {len(sp)} AMPs vs {len(sn)} non-AMPs  (total {len(seqs)})')

## 4. Featurize the sequences
A peptide is a string like `GIGKFLHSAKKFGKAFVGEIMNS`. Models need numbers. We use cheap, interpretable, biology-motivated features:
- **Amino-acid composition** (fraction of each of the 20 amino acids) — 20 features.
- **Physicochemical descriptors**: length, net charge, charge density, mean hydrophobicity (Kyte–Doolittle), fraction hydrophobic/polar/positive/negative, aromaticity.

We deliberately start simple (no deep learning yet) — a strong, fast, explainable baseline.

In [ ]:
KD={'A':1.8,'R':-4.5,'N':-3.5,'D':-3.5,'C':2.5,'Q':-3.5,'E':-3.5,'G':-0.4,'H':-3.2,
    'I':4.5,'L':3.8,'K':-3.9,'M':1.9,'F':2.8,'P':-1.6,'S':-0.8,'T':-0.7,'W':-0.9,'Y':-1.3,'V':4.2}
def featurize(s):
    L=len(s); c=Counter(s)
    aac=[c.get(a,0)/L for a in CANON]
    charge=c.get('K',0)+c.get('R',0)+0.1*c.get('H',0)-c.get('D',0)-c.get('E',0)
    return aac+[L, charge, charge/L, sum(KD[a] for a in s)/L,
                sum(c.get(a,0) for a in 'AILMFWVC')/L, sum(c.get(a,0) for a in 'STNQ')/L,
                (c.get('K',0)+c.get('R',0)+c.get('H',0))/L, (c.get('D',0)+c.get('E',0))/L,
                sum(c.get(a,0) for a in 'FWY')/L]
FEAT=list(CANON)+['length','net_charge','charge_density','mean_hydrophobicity',
                  'frac_hydrophobic','frac_polar','frac_pos','frac_neg','aromaticity']
X=np.array([featurize(s) for s in seqs]); print('feature matrix:', X.shape)

## 5. Train + honestly evaluate
5-fold stratified cross-validation (every peptide is in the test fold exactly once). We report ROC-AUC, accuracy, F1 and MCC (Matthews correlation — robust on balanced binary tasks).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, matthews_corrcoef, f1_score, confusion_matrix

cv=StratifiedKFold(5, shuffle=True, random_state=42)
def evaluate(model,name):
    p=cross_val_predict(model,X,y,cv=cv,method='predict_proba')[:,1]; pr=(p>=.5).astype(int)
    print(f'[{name}]  AUC={roc_auc_score(y,p):.3f}  Acc={accuracy_score(y,pr):.3f}  '
          f'F1={f1_score(y,pr):.3f}  MCC={matthews_corrcoef(y,pr):.3f}  cm={confusion_matrix(y,pr).tolist()}')
    return roc_auc_score(y,p)
logreg=make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
rf=RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1)
a1=evaluate(logreg,'LogReg'); a2=evaluate(rf,'RandomForest')

## 6. What did the model learn? (biological sanity check)
If the top features are net charge / cationic residues, the model learned the real mechanism: AMPs are positively charged and disrupt negatively-charged bacterial membranes. That's the result we want.

In [ ]:
rf.fit(X,y)
imp=sorted(zip(FEAT, rf.feature_importances_), key=lambda t:-t[1])[:12]
for n,v in imp: print(f'  {n:18s} {v:.3f}')
plt.figure(figsize=(7,3.6)); names=[n for n,_ in imp][::-1]; vals=[v for _,v in imp][::-1]
plt.barh(names, vals); plt.title('Top features driving the AMP prediction'); plt.tight_layout(); plt.show()

## 7. Save the model + score new peptides
Reusable scorer for later layers (the generator's candidates flow through this).

In [ ]:
best = rf if a2>=a1 else logreg
best.fit(X,y)
joblib.dump({'model':best,'features':FEAT,'minlen':MINLEN,'maxlen':MAXLEN}, 'amp_activity_baseline.joblib')

def amp_score(seq):
    seq=seq.upper()
    return float(best.predict_proba(np.array([featurize(seq)]))[0,1])

tests={'Magainin-2 (known AMP)':'GIGKFLHSAKKFGKAFVGEIMNS',
       'Melittin (known AMP)':'GIGAVLKVLTTGLPALISWIKRKRQQ',
       'Random-ish non-AMP':'SEEGDTAATGGDSTGAESDTAAGSE'}
for n,s in tests.items(): print(f'  {amp_score(s):.2f}   {n}')

## 8. Limitations & what Layer 2 adds (be honest in the writeup)
- **Negatives are a modelling choice.** UniProt decoys answer *"is this an AMP at all?"*. A different, harder question — *"among real peptides, which are most potent?"* — uses GRAMPA's MIC directly (regression on the `value` column / a low-vs-high-MIC split). Both are worth building; this is the single biggest assumption to stress-test.
- **No homology de-duplication yet.** Near-identical sequences can inflate scores. Next: cluster with CD-HIT / MMseqs2 and split by cluster.
- **Composition features ignore residue order.** A sequence model (CNN/ESM-2 embeddings) is the upgrade path.

**Layer 2:** MIC regression (predict the number, not just yes/no) + the toxicity/hemolysis filter (Hemolytik/DRAMP). **Layer 3:** the generator. Then assemble the loop.